In [16]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef

# RQ4: Does the use of a voting-based ensemble method enhance predictive performance?

In [17]:
rq1_withoutMsg_results_path = "./Results/RQ1/WithoutMsg"
rq1_withMsg_results_path = "./Results/RQ1/WithMsg"

rq2_cot_results_path = "./Results/RQ2/COT"
rq2_fewshot_results_path = "./Results/RQ2/FewShot"
rq2_fewshotCot_results_path = "./Results/RQ2/FewShotCOT"

rq3_results_path = "./Results/RQ3"

## Ensemble all models

In [18]:
rq1_llms = ["gemini-2.0-flash", "gpt-4o", "gpt-4o-mini"]
rq2_llms = ["gemini-2.0-flash", "gpt-4o"]
rq3_models = ["RF", "DT", "MLP", "SVC"]

In [19]:
all_resutls_path = []
for model in rq1_llms:
    for results_path in [rq1_withoutMsg_results_path, rq1_withMsg_results_path]:
        results_path = os.path.join(results_path, model + ".csv")
        all_resutls_path.append(results_path)

for model in rq2_llms:
    for results_path in [rq2_cot_results_path, rq2_fewshot_results_path, rq2_fewshotCot_results_path]:
        results_path = os.path.join(results_path, model + ".csv")
        all_resutls_path.append(results_path)

for model in rq3_models:
    results_path = os.path.join(rq3_results_path, model + ".csv")
    all_resutls_path.append(results_path)


In [ ]:
results = {}
models = []

for result_path in all_resutls_path:
    model = "/".join(result_path.split("/")[2:]).replace(".csv", "")
    models.append(model)

    df = pd.read_csv(result_path)
    results["Decision"] = df["Decision"].tolist()
    results[model + "_Detection"] = df["Detection"].tolist()

df = pd.DataFrame(results)
ensemble_results = []
for indx, row in df.iterrows():
    buggy = 0
    not_buggy = 0
    for model in models:
        if row[model + "_Detection"] == "Buggy":
            buggy += 1
        else:
            not_buggy += 1
    if buggy >= not_buggy:
        ensemble_results.append("Buggy")
    else:
        ensemble_results.append("NotBuggy")
df["Ensemble_Detection"] = ensemble_results

results = []
for model in models + ["Ensemble"]:
    row = {
        "Model": model,
        "Accuracy": accuracy_score(df["Decision"], df[model + "_Detection"]),
        "Precision": precision_score(df["Decision"], df[model + "_Detection"], pos_label="Buggy"),
        "Recall": recall_score(df["Decision"], df[model + "_Detection"], pos_label="Buggy"),
        "F1": f1_score(df["Decision"], df[model + "_Detection"], pos_label="Buggy"),
        "MCC": matthews_corrcoef(df["Decision"], df[model + "_Detection"]),
    }
    results.append(row)

metrics_df = pd.DataFrame(results)

In [22]:
metrics_df

,Model,Accuracy,Precision,Recall,F1,MCC
0,RQ1/WithoutMsg/gemini-2.0-flash,0.692177,0.650213,0.852679,0.737808,0.401956
1,RQ1/WithMsg/gemini-2.0-flash,0.795918,0.717180,0.987723,0.830986,0.638640
2,RQ1/WithoutMsg/gpt-4o,0.691043,0.654083,0.831473,0.732187,0.394633
3,RQ1/WithMsg/gpt-4o,0.873583,0.867760,0.886161,0.876864,0.747189
4,RQ1/WithoutMsg/gpt-4o-mini,0.704082,0.660103,0.860491,0.747093,0.426090
5,RQ1/WithMsg/gpt-4o-mini,0.821995,0.763587,0.940848,0.843000,0.661369
6,RQ2/COT/gemini-2.0-flash,0.765306,0.687988,0.984375,0.809917,0.587355
7,RQ2/FewShot/gemini-2.0-flash,0.802154,0.723265,0.988839,0.835455,0.649323
8,RQ2/FewShotCOT/gemini-2.0-flash,0.806689,0.731443,0.978795,0.837232,0.651268
9,RQ2/COT/gpt-4o,0.826531,0.769653,0.939732,0.846231,0.668939


## Enseble best models

In [23]:
all_resutls_path = [
        os.path.join(rq1_withMsg_results_path, "gpt-4o.csv"),
        os.path.join(rq2_fewshotCot_results_path, "gpt-4o.csv"),
        os.path.join(rq3_results_path, "MLP.csv"),
    ]

In [24]:
results = {}
models = []

for result_path in all_resutls_path:
    model = "/".join(result_path.split("/")[2:]).replace(".csv", "")
    models.append(model)

    df = pd.read_csv(result_path)
    results["Decision"] = df["Decision"].tolist()
    results[model + "_Detection"] = df["Detection"].tolist()

df = pd.DataFrame(results)
ensemble_results = []
for indx, row in df.iterrows():
    buggy = 0
    not_buggy = 0
    for model in models:
        if row[model + "_Detection"] == "Buggy":
            buggy += 1
        else:
            not_buggy += 1
    if buggy >= not_buggy:
        ensemble_results.append("Buggy")
    else:
        ensemble_results.append("NotBuggy")
df["Ensemble_Detection"] = ensemble_results

results = []
for model in models + ["Ensemble"]:
    row = {
        "Model": model,
        "Accuracy": accuracy_score(df["Decision"], df[model + "_Detection"]),
        "Precision": precision_score(df["Decision"], df[model + "_Detection"], pos_label="Buggy"),
        "Recall": recall_score(df["Decision"], df[model + "_Detection"], pos_label="Buggy"),
        "F1": f1_score(df["Decision"], df[model + "_Detection"], pos_label="Buggy"),
        "MCC": matthews_corrcoef(df["Decision"], df[model + "_Detection"]),
    }
    results.append(row)

metrics_df = pd.DataFrame(results)

In [25]:
metrics_df

,Model,Accuracy,Precision,Recall,F1,MCC
0,RQ1/WithMsg/gpt-4o,0.873583,0.867760,0.886161,0.876864,0.747189
1,RQ2/FewShotCOT/gpt-4o,0.879819,0.870933,0.896205,0.883388,0.759795
2,RQ3/MLP,0.903628,0.898026,0.914062,0.905973,0.807285
3,Ensemble,0.901927,0.894220,0.915179,0.904578,0.803959
